# 🐛 DebugZero — GRPO Training on Colab

Two modes available:
- **Offline** — builds a static bug bank, trains GRPO on it (fast, simple)
- **Online** *(AZR-inspired)* — model generates its own bugs each iteration (self-play)

**Recommended GPU:** T4 (free tier) or A100 (Colab Pro)

## 1. Setup & Install

In [1]:
# Clone the repo (skip if already cloned)
import os
if not os.path.exists('DebugZero'):
    !git clone -b side2 https://github.com/Ray-0906/DebugZero.git
os.chdir('DebugZero')
print('Working dir:', os.getcwd())

Working dir: /content/DebugZero


In [2]:
%%capture
!pip install -q unsloth
!pip install -q trl transformers datasets openenv-core[core] pydantic thefuzz matplotlib bitsandbytes vllm

In [3]:
# Verify GPU is available
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → GPU')

CUDA available: True
GPU: Tesla T4
Memory: 15.6 GB


## 2. Configuration

In [4]:
# === Training Mode ===
ONLINE_MODE = False   # False = offline GRPO (recommended to start)
DRY_RUN = False       # True = tiny smoke test (no real model)

# === Online mode settings (ignored if ONLINE_MODE=False) ===
ITERATIONS = 3        # Number of self-play iterations
STEPS_PER_ITER = 30   # GRPO steps per iteration

In [5]:
# Temporarily disable vLLM's torch.compile to avoid graph compilation issues
import os
os.environ["VLLM_NO_COMPILE"] = "1"
print("VLLM_NO_COMPILE set to 1 to disable torch.compile for vLLM.")

VLLM_NO_COMPILE set to 1 to disable torch.compile for vLLM.


In [6]:
import unsloth

# Forcefully reset to a clean state if we already patched it
if hasattr(unsloth.FastLanguageModel, '_true_original_from_pretrained'):
    unsloth.FastLanguageModel.from_pretrained = unsloth.FastLanguageModel._true_original_from_pretrained

# Save the clean original reference
unsloth.FastLanguageModel._true_original_from_pretrained = unsloth.FastLanguageModel.from_pretrained

def patched_from_pretrained_v3(*args, **kwargs):
    # Ensure compilation_config is None to stop vLLM from attempting to compile kernels
    kwargs['compilation_config'] = None
    return unsloth.FastLanguageModel._true_original_from_pretrained(*args, **kwargs)

unsloth.FastLanguageModel.from_pretrained = patched_from_pretrained_v3
print("Reset and re-patched FastLanguageModel.from_pretrained to disable vLLM compilation.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Reset and re-patched FastLanguageModel.from_pretrained to disable vLLM compilation.


## 3. Verify Data

In [7]:
from training.grpo_train import create_dataset

dataset, bug_bank = create_dataset()
print(f'✅ Dataset rows:    {len(dataset)}')
print(f'✅ Training bugs:   {len(bug_bank.train_samples)}')
print(f'✅ Eval bugs:       {len(bug_bank.eval_samples)}')
print(f'\nSample row:')
dataset[0]

Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.
✅ Dataset rows:    27
✅ Training bugs:   18
✅ Eval bugs:       6

Sample row:


{'prompt': 'You are the Solver in a debugging self-play game.\nFix the bug with the smallest correct local change and return only the full fixed Python code inside triple backticks.\n\nBuggy function:\n```python\ndef has_close_elements(numbers: list[float], threshold: float) -> bool:\n    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx == idx2:\n                distance = abs(elem - elem2)\n                if distance < threshold:\n                    return True\n    return False\n```\n\nFailure summary:\nTraceback (most recent call last):\n^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\nAssertionError\n',
 'metadata': {'role': 'solver',
  'seed_id': 'HumanEval/0',
  'original_code': 'def has_close_elements(numbers: list[float], threshold: float) -> bool:\n    for idx, elem in enumerate(numbers):\n        for idx2, elem2 in enumerate(numbers):\n            if idx != idx2:\n                distance = abs(elem - elem2)\n                

## 4. Train

In [8]:
if ONLINE_MODE:
    from training.online_grpo_train import run_online_workflow
    results = run_online_workflow(
        dry_run=DRY_RUN,
        iterations=ITERATIONS,
        steps_per_iter=STEPS_PER_ITER,
    )
else:
    from training.grpo_train import run_workflow
    results = run_workflow(dry_run=DRY_RUN)

print('\n✅ Training complete!')

Built dataset with 27 rows from 18 training bugs and 6 eval bugs.
Initializing Unsloth FastLanguageModel...
INFO 04-25 13:56:08 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.6.2. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-3b-instruct-bnb-4bit with actual GPU utilization = 49.41%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 1024. Num Sequences = 32.
Unsloth: vLLM's KV Cache can use up to 4.63 GB. Also swap space = 0 GB.
Unsloth: Not

<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


INFO 04-25 13:56:20 [weight_utils.py:625] No model.safetensors.index.json found in remote.


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 04-25 13:56:22 [punica_selector.py:20] Using PunicaWrapperGPU.
INFO 04-25 13:56:23 [gpu_model_runner.py:4820] Model loading took 2.17 GiB memory and 3.484883 seconds
INFO 04-25 13:56:43 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/00d677c82b/rank_0_0/backbone for vLLM's torch.compile
INFO 04-25 13:56:43 [backends.py:1111] Dynamo bytecode transform time: 18.71 s


RuntimeError: Tried to erase Node size_3 but it still had 1 users in the graph: {buffer: None}!

## 5. Results

In [ ]:
print('=== Pre-Training ===')
print(f"  Solver:   {results['pre_solver_metrics']}")
print(f"  Proposer: {results['pre_proposer_metrics']}")
print()
print('=== Post-Training ===')
print(f"  Solver:   {results['post_solver_metrics']}")
print(f"  Proposer: {results['post_proposer_metrics']}")

# Online mode: show per-iteration metrics
if ONLINE_MODE and 'iteration_metrics' in results:
    print('\n=== Per-Iteration ===')
    for m in results['iteration_metrics']:
        print(f"  Iter {m['iteration']}: pool={m['pool_size']} ds={m['dataset_size']} "
              f"solver={m['solver']} proposer={m['proposer']}")

In [ ]:
from IPython.display import Image, display

if results.get('plot_path'):
    display(Image(filename=results['plot_path']))
else:
    print('No plot generated.')

## 6. Save Model (Optional)

In [ ]:
# Save to Google Drive (optional)
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r debugzero_model /content/drive/MyDrive/debugzero_model
    print('✅ Model saved to Google Drive')
else:
    print('Model saved locally at: debugzero_model/')
    print('Set SAVE_TO_DRIVE = True above to persist to Google Drive.')